# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: 1 row = 1 unique page URL identified by content_id for a specific client_id
Dataset Table: content_refresh_anonymized.csv (mid-panel starter slice)
Time Window: Historical content performance spanning ages 90 to 564 days
Target Proxy: is_declining_label (1 if trend_direction == 'down', 0 otherwise)
Deliberate Exclusion: trend_pct (future percentage drop metric)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature Fields: content_age_days, days_since_last_update, avg_position, ctr, impressions_90d
Label Field: is_declining_label (binary flag for performance decline)
Context Fields: content_id, client_id, content_type, main_intent
Excluded Fields: trend_pct — Why: It explicitly measures the future percentage change in traffic after the decision moment Using it introduces data leakage by handing the AI the cheat answer

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

is_unique = df['content_id'].nunique() == len(df)
print(f"Query 1 (Grain): Is 1 row == 1 unique content ID? {is_unique} ({len(df):,} total rows)")

min_age = df['content_age_days'].min()
max_age = df['content_age_days'].max()
print(f"Query 2 (Scale & Span): {len(df):,} rows spanning content age from {min_age} to {max_age} days.")

available_mask = df['ctr'].notna() & df['avg_position'].notna() & df['impressions_90d'].notna()
surviving_rows = available_mask.sum()
print(f"Query 3 (Availability): {surviving_rows:,} rows survive missing value filtering (100% complete).")

features = [
    'content_age_days',       
    'days_since_last_update', 
    'avg_position',           
    'ctr',                    
    'impressions_90d'         
]

clean_df = df[features + ['is_declining_label', 'trend_pct']].dropna()

clean_df['TRAP_trend_pct'] = clean_df['trend_pct'] 

X_trap = clean_df[features + ['TRAP_trend_pct']]
X_honest = clean_df[features]
y = clean_df['is_declining_label']

X_tr_t, X_te_t, y_tr, y_te = train_test_split(X_trap, y, test_size=0.2, random_state=42)
X_tr_h, X_te_h, _, _ = train_test_split(X_honest, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)

model.fit(X_tr_t, y_tr)
trap_acc = accuracy_score(y_te, model.predict(X_te_t))

model.fit(X_tr_h, y_tr)
honest_acc = accuracy_score(y_te, model.predict(X_te_h))

print(f"\nTRAP SCORE (with leaked trend_pct): {trap_acc:.4f} (Deceptively near-perfect score)")
print(f"HONEST SCORE (after dropping trap column): {honest_acc:.4f} (Real baseline score)")

Query 1 (Grain): Is 1 row == 1 unique content ID? True (30,000 total rows)
Query 2 (Scale & Span): 30,000 rows spanning content age from 90 to 564 days.
Query 3 (Availability): 30,000 rows survive missing value filtering (100% complete).

TRAP SCORE (with leaked trend_pct): 0.9996 (Deceptively near-perfect score)
HONEST SCORE (after dropping trap column): 0.6727 (Real baseline score)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Competitor Blindness: This dataset only records internal client metrics (impressions, CTR, content age). It cannot observe competitor activity, market shifts, or search engine algorithm updates that directly drive traffic drops.

Single Historical Snapshot: The dataset captures a static point-in-time snapshot rather than continuous, real-time streaming data, limiting our ability to track daily intra-week fluctuations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.